In [1]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [ ]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [3]:

queryOderDetail = """
SELECT 
[SalesOrderID]
      ,[CarrierTrackingNumber]
      ,[OrderQty]
      ,[ProductID]
      ,[SpecialOfferID]
      ,[UnitPrice]
      ,[UnitPriceDiscount]
      ,[LineTotal]
FROM Sales.SalesOrderDetail
"""

tablaSalesOrderDetail = pd.read_sql_query(queryOderDetail, motorBaseDatos)




queryOderHeader = """
SELECT 
[SalesOrderID]
      ,[RevisionNumber]
      ,[OrderDate]
      ,[DueDate]
      ,[ShipDate]
      ,[SalesOrderNumber]
      ,[CustomerID]
      ,[SalesPersonID]
      ,[TerritoryID]
      ,[TaxAmt]
      ,[Freight]
FROM Sales.SalesOrderHeader
"""

tablaSalesOrderHeader = pd.read_sql_query(queryOderHeader, motorBaseDatos)



queryProduct = """
SELECT 
[ProductID]
      ,[StandardCost]
FROM Production.Product
"""

tablaProduct = pd.read_sql_query(queryProduct, motorBaseDatos)


# tablaSalesOrderDetail
# tablaSalesOrderHeader
# dimensionProduct
# tablaSpecialOffert
# tablaSpecialOffertProduct
# tablaProduct

In [4]:
# tablaSalesOrderDetail


In [5]:
tablaSalesOrderDetail = tablaSalesOrderDetail.merge(tablaProduct, on='ProductID')
# tablaSalesOrderDetail

In [6]:
# tablaSalesOrderHeader


TRANSFORMACION

In [7]:
tablaSales = tablaSalesOrderDetail.merge(tablaSalesOrderHeader, on='SalesOrderID')

# tablaSales

In [8]:

tablaSales.rename(columns={
    'TerritoryID': 'SalesTerritoryKey',
    'CustomerID': 'CustomerKey',
    'UnitPriceDiscount': 'UnitPriceDiscountPct',
    'SpecialOfferID': 'PromotionKey',
    'OrderQty': 'OrderQuantity',
    'LineTotal' : 'ExtendedAmount',
    'StandardCost' : 'ProductStandardCost',
    'ProductID' : 'ProductKey'
}, inplace=True)


tablaSales["DiscountAmount"] = tablaSales["UnitPrice"] * tablaSales["UnitPriceDiscountPct"] * tablaSales["OrderQuantity"] 

tablaSales["TotalProductCost"] = tablaSales["ProductStandardCost"] * tablaSales["OrderQuantity"] 

tablaSales["SalesAmount"] = tablaSales["ExtendedAmount"]

tablaSales["CustomerPONumber"] = None
tablaSales["CurrencyKey"] = None
tablaSales["SalesOrderLineNumber"] = None

tablaSales["OrderDateKey"] = pd.to_datetime(tablaSales["OrderDate"]).dt.strftime('%Y%m%d')
tablaSales["DueDateKey"] = pd.to_datetime(tablaSales["DueDate"]).dt.strftime('%Y%m%d')
tablaSales["ShipDateKey"] = pd.to_datetime(tablaSales["ShipDate"]).dt.strftime('%Y%m%d')

tablaSales.drop(columns=[
    'SalesOrderID',
    'SalesPersonID',
], inplace=True)

# tablaSales

CARGAR A LA BODEGA

In [9]:
tablaSales.to_sql('hechoInternetSales',motorBodegaDatos, if_exists='replace',index=False)

37